# PX1: проверка образа SD-карты Mini-Cam с Android

Этот блокнот запускается в Google Colab на сервере Google: 2,93 ГБ не скачиваются на телефон.

Нажми «Среда выполнения → Выполнить все» и предоставь Google Colab доступ к своему Диску. Блокнот сохранит небольшой текстовый отчёт PX1_SD_REPORT.txt в корень твоего Google Диска. Потом напиши в ChatGPT: «Готово, прочитай отчёт».

Это предварительная офлайн-диагностика, а не подтверждение каких-либо уязвимостей. Образ используется только для чтения, не загружается в GitHub и не модифицируется. Для анализа применяется сторонняя утилита [unimgc](https://github.com/Shizmob/unimgc). Перед подключением Google Диска проверь доступ и код блокнота.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
known = root / 'Untitled folder' / 'ЛП.imgc'
if known.is_file():
    image_path = known
else:
    image_path = None
    for current, dirs, files in os.walk(root):
        if 'ЛП.imgc' in files:
            image_path = Path(current) / 'ЛП.imgc'
            break
if image_path is None:
    raise FileNotFoundError('ЛП.imgc не найден. Проверь подключённый Google-аккаунт.')
print('Найден:', image_path.name, '| байт:', image_path.stat().st_size)


In [ ]:
import subprocess
from pathlib import Path
repo = Path('/content/unimgc')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Shizmob/unimgc.git', str(repo)], check=True, timeout=180)
subprocess.run(['make', '-C', str(repo)], check=True, timeout=120)
unimgc = repo / 'unimgc'
if not unimgc.is_file():
    raise FileNotFoundError('Не удалось собрать unimgc')
res = subprocess.run([str(unimgc), '-i', str(image_path)], capture_output=True, text=True, timeout=180)
print('Проверка заголовка завершена, код:', res.returncode)
# Не выводим сырой заголовок: там могут быть идентификаторы накопителя.


In [ ]:
import struct, subprocess
from pathlib import Path
N = 16 * 1024 * 1024
proc = subprocess.Popen([str(unimgc), str(image_path)], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
with proc.stdout:
    head = proc.stdout.read(N)
proc.terminate()
try:
    proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait(timeout=10)

def fs_at(buf, ofs):
    if ofs < 0 or ofs+2048 > len(buf):
        return 'вне считанного фрагмента'
    s = buf[ofs:ofs+4096]
    if s[3:11] == b'NTFS    ': return 'NTFS'
    if s[3:11] == b'EXFAT   ': return 'exFAT'
    if b'FAT32' in s[64:95]: return 'FAT32'
    if b'FAT16' in s[32:95]: return 'FAT16'
    if s[1080:1082] == b'\x53\xef': return 'ext2/3/4 (signature)'
    return 'не определена'

lines = ['PX1 / Mini-Cam — первичный анализ SD-образа (не заключение об уязвимостях)',
         'Файл: ' + image_path.name,
         'Сжатый размер: ' + str(image_path.stat().st_size),
         'Считано распакованных байт: ' + str(len(head)),
         'Первые 16 байт: ' + head[:16].hex()]
if len(head) >= 512 and head[510:512] == b'\x55\xaa':
    lines.append('MBR/boot signature: 55AA')
    for i in range(4):
        e = head[446 + 16*i:446 + 16*(i+1)]
        typ = e[4]
        lba, count = struct.unpack_from('<II', e, 8)
        if typ or count:
            lines.append('Раздел %s: тип 0x%02x, начало LBA %s, секторов %s, FS: %s' % (i+1, typ, lba, count, fs_at(head, lba*512)))
else:
    lines.append('MBR-сигнатура не выявлена')
lines.append('FS в начале raw-образа: ' + fs_at(head,0))
lines.append('IMGC заголовок: ' + ('OK' if res.returncode == 0 else 'ошибка проверки'))
lines.append('Важно: первые 16 МБ недостаточны для окончательного вывода о наличии ОС.')
out = Path('/content/drive/MyDrive/PX1_SD_REPORT.txt')
out.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines))
print('ГОТОВО: PX1_SD_REPORT.txt сохранён в корне Google Диска. Вернись в ChatGPT.')
